# 02 - Daily Statistics Chart

This notebook generates a stacked area chart showing daily on-time/minor/major delay breakdown.

**Features:**
- Stacked area chart with daily counts
- On-time percentage overlay line
- Interactive date range selection

**Output:** `static/plots/daily_stats.html`

In [13]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
OUTPUT_DIR = os.path.join(project_root, 'static', 'plots')

print(f"Database exists: {os.path.exists(DB_PATH)}")

Database exists: True


## Configuration

In [14]:
# ====================
# CONFIGURATION
# ====================

# Date range (None = all data)
DAYS_TO_SHOW = 60

# Colors
STATUS_COLORS = {
    'On Time': '#00CC96',
    'Minor': '#FECB52',
    'Major': '#EF553B',
}

THEME = {
    'bg_color': '#0d1117',
    'text_color': '#c9d1d9',
    'grid_color': '#21262d',
}

FIGURE_WIDTH = 1000
FIGURE_HEIGHT = 500

## Load and Process Data

In [15]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_arrivals():
    conn = sqlite3.connect(DB_PATH)
    
    date_filter = ""
    if DAYS_TO_SHOW:
        cutoff = (datetime.now() - timedelta(days=DAYS_TO_SHOW)).strftime('%Y-%m-%d')
        date_filter = f" WHERE timestamp >= '{cutoff}'"
    
    query = f"SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp FROM train_locations {date_filter}"
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']), axis=1)
    df['date'] = df['timestamp'].dt.date
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1)
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_arrivals()

Processed 99,791 arrivals


## Calculate Daily Statistics

In [16]:
def calculate_daily_stats(arrivals):
    if arrivals.empty:
        return pd.DataFrame()
    
    # Count by status per day
    daily = arrivals.groupby(['date', 'status']).size().reset_index(name='count')
    pivot = daily.pivot(index='date', columns='status', values='count').fillna(0)
    
    for s in ['On Time', 'Minor', 'Major']:
        if s not in pivot.columns:
            pivot[s] = 0
    
    pivot = pivot[['On Time', 'Minor', 'Major']]
    pivot['total'] = pivot.sum(axis=1)
    pivot['on_time_pct'] = (pivot['On Time'] / pivot['total'] * 100).round(1)
    pivot = pivot.reset_index()
    pivot['date'] = pd.to_datetime(pivot['date'])
    pivot = pivot.sort_values('date')
    
    return pivot

daily_stats = calculate_daily_stats(arrivals)
daily_stats.tail(10)

status,date,On Time,Minor,Major,total,on_time_pct
49,2025-12-19,1628.0,69.0,189.0,1886.0,86.3
50,2025-12-20,746.0,78.0,0.0,824.0,90.5
51,2025-12-21,1330.0,49.0,0.0,1379.0,96.4
52,2025-12-22,1807.0,100.0,19.0,1926.0,93.8
53,2025-12-23,1832.0,99.0,0.0,1931.0,94.9
54,2025-12-24,1369.0,152.0,3.0,1524.0,89.8
55,2025-12-25,1185.0,196.0,2.0,1383.0,85.7
56,2025-12-26,1799.0,137.0,3.0,1939.0,92.8
57,2025-12-27,1371.0,33.0,0.0,1404.0,97.6
58,2025-12-28,231.0,21.0,21.0,273.0,84.6


## Create Chart

In [17]:
def create_daily_chart(daily_stats):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Stacked areas
    for status in ['On Time', 'Minor', 'Major']:
        fig.add_trace(
            go.Scatter(
                x=daily_stats['date'],
                y=daily_stats[status],
                name=status,
                fill='tonexty' if status != 'On Time' else 'tozeroy',
                mode='none',
                fillcolor=STATUS_COLORS[status],
                stackgroup='one',
                hovertemplate=f'{status}: %{{y:,.0f}}<extra></extra>',
            ),
            secondary_y=False,
        )
    
    # On-time percentage line
    fig.add_trace(
        go.Scatter(
            x=daily_stats['date'],
            y=daily_stats['on_time_pct'],
            name='On-Time %',
            mode='lines',
            line=dict(color='#58a6ff', width=2, dash='dot'),
            hovertemplate='On-Time: %{y:.1f}%<extra></extra>',
        ),
        secondary_y=True,
    )
    
    # Summary
    total = daily_stats['total'].sum()
    avg_pct = daily_stats['on_time_pct'].mean()
    date_range = f"{daily_stats['date'].min().strftime('%b %d')} - {daily_stats['date'].max().strftime('%b %d, %Y')}"
    
    fig.update_layout(
        title=dict(
            text=f'📊 Daily Performance Breakdown<br><span style="font-size:14px;color:{THEME["text_color"]}">{date_range} | {int(total):,} arrivals | {avg_pct:.1f}% avg on-time</span>',
            font=dict(size=22, color=THEME['text_color']),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            title='Date',
            tickfont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
        ),
        yaxis=dict(
            title='Number of Arrivals',
            tickfont=dict(color=THEME['text_color']),
            # titlefont=dict(color=THEME['text_color']),
            gridcolor=THEME['grid_color'],
        ),
        yaxis2=dict(
            title='On-Time %',
            ticksuffix='%',
            tickfont=dict(color='#58a6ff'),
            # titlefont=dict(color='#58a6ff'),
            range=[0, 100],
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=-0.2,
            xanchor='center',
            x=0.5,
            font=dict(color=THEME['text_color']),
        ),
        plot_bgcolor=THEME['bg_color'],
        paper_bgcolor=THEME['bg_color'],
        height=FIGURE_HEIGHT,
        width=FIGURE_WIDTH,
        margin=dict(l=60, r=60, t=100, b=80),
        hovermode='x unified',
    )
    
    return fig

fig = create_daily_chart(daily_stats)
fig.show()

## Export

In [18]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'daily_stats.html')
fig.write_html(output_path, include_plotlyjs='cdn')
print(f"✅ Exported to: {output_path}")

✅ Exported to: d:\caltrain-tracker\static\plots\daily_stats.html
